[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C49_Encoder_Seq2Seq_Course/02_pretraining_objectives/02_pretraining_objectives.ipynb)

# 02 · 预训练目标与配方的改良（RoBERTa / ELECTRA / DeBERTa / ALBERT）

目标：把 **静态 vs 动态掩码 → RTD 判别式目标 → 解耦注意力 → 参数共享与嵌入分解** 从零实现，
每项改进都**量化它到底改善了什么指标**。

路线：动态掩码的有效样本量 → RTD 的信号密度与信息量 → 生成器规模的「难度匹配」→
DeBERTa 三项注意力分解 → ALBERT 的参数量 vs FLOPs → 四维度对比 → ✏️ 练习 → 📖 答案 → 🧪 效率前沿胶囊。

> 心智模型：**配方 / 目标 / 位置信息 / 参数放置 是四个几乎正交的维度**，
> 每个优化的是不同指标。问「谁最好」是坏问题，问「我被什么卡住」才对。

## 1 · RoBERTa 的动态掩码：免费的数据增强

BERT 在预处理阶段就固定了掩码（做了 10 份副本），40 个 epoch 里反复看同样的挖空。
RoBERTa 每次现掩。先量化这个差别。

In [ ]:
import numpy as np, math, itertools
rng = np.random.default_rng(0)

L, MASK_RATE = 20, 0.15
K_MASK = int(round(L * MASK_RATE))

def static_patterns(n_copies, seed=0):
    '''BERT：预处理时生成 n_copies 份固定掩码，训练时循环复用。'''
    r = np.random.default_rng(seed)
    return [frozenset(r.choice(L, size=K_MASK, replace=False)) for _ in range(n_copies)]

def dynamic_patterns(n_epochs, seed=0):
    '''RoBERTa：每个 epoch 现场采样。'''
    r = np.random.default_rng(seed)
    return [frozenset(r.choice(L, size=K_MASK, replace=False)) for _ in range(n_epochs)]

N_EPOCHS = 40
static_seen  = set(static_patterns(10))          # BERT 论文：10 份静态副本
dynamic_seen = set(dynamic_patterns(N_EPOCHS))
print(f'序列长度 {L}, 掩 {K_MASK} 个位置, 训练 {N_EPOCHS} 个 epoch')
print(f'静态(10份副本): 模型见过 {len(static_seen)} 种不同的挖空模式')
print(f'动态(每轮现掩): 模型见过 {len(dynamic_seen)} 种不同的挖空模式')
print(f'理论上限 C({L},{K_MASK}) = {math.comb(L, K_MASK):,}')

assert len(static_seen) <= 10, '静态副本数是硬上限'
assert len(dynamic_seen) > len(static_seen), '动态掩码见到的模式更多'
print(f'\n✅ 动态掩码把有效「样本」数提高了 {len(dynamic_seen)/len(static_seen):.1f} 倍。')
print('   注意：实际收益远小于组合数上限（不同挖空高度相关），但方向明确 ——')
print('   **训练时的随机性是一种免费的数据增强**。')

### RoBERTa 改动清单的「贡献归因」

论文的价值在消融表。用一个简化模型复现这个归因逻辑：**每项改动的边际贡献**。

In [ ]:
# 数字取自 RoBERTa 论文报告的量级（GLUE 平均分）
ablation = [
    ('BERT-base 复现基线',           79.0),
    ('+ 去掉 NSP、改用整句输入',      80.5),
    ('+ 动态掩码',                    81.2),
    ('+ 更大 batch (256 -> 8k)',      82.4),
    ('+ 更多数据 (16GB -> 160GB)',    84.6),
    ('+ 更长训练',                    86.0),
]
prev = None
print(f"{'配置':<32s} {'GLUE':>6s} {'边际':>6s}")
for name, score in ablation:
    delta = '' if prev is None else f'{score - prev:+.1f}'
    print(f'{name:<32s} {score:>6.1f} {delta:>6s}')
    prev = score

deltas = [ablation[i][1] - ablation[i-1][1] for i in range(1, len(ablation))]
assert all(d > 0 for d in deltas), '每一项都应有正贡献'
biggest = max(range(len(deltas)), key=lambda i: deltas[i])
print(f'\n最大单项贡献: 「{ablation[biggest+1][0]}」 +{deltas[biggest]:.1f}')
assert ablation[-1][1] - ablation[0][1] > 6, '总提升应超过 6 分'
print(f'✅ 总提升 {ablation[-1][1]-ablation[0][1]:.1f} 分，**零架构改动**。')
print('   教训：宣布架构创新有效之前，先确认 baseline 是充分训练的。')

## 2 · ELECTRA 的 RTD：每个位置都有信号

生成器（小 MLM）造替换 → 判别器判断每个 token 是否被替换。
**信号密度从 0.15 拉回 1.0，且输入里没有 `[MASK]`。**

In [ ]:
V, D = 40, 24

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def make_generator(strength, seed=0):
    '''一个「玩具生成器」：strength 越大，替换越像真词（越难分辨）。
       用一个随机的 token->token 相似度矩阵模拟。'''
    r = np.random.default_rng(seed)
    sim = r.normal(size=(V, V)) * 0.5
    sim += np.eye(V) * strength          # strength 大 -> 更倾向于生成「接近原词」的替换
    return softmax(sim, axis=-1)

def electra_corrupt(ids, gen_probs, mask_rate=0.15, seed=0):
    '''返回 (被污染的 ids, RTD 标签 1=被替换 0=原始)。**注意：没有 [MASK]**'''
    r = np.random.default_rng(seed)
    ids = ids.copy()
    labels = np.zeros(len(ids), dtype=int)
    k = max(1, int(round(len(ids) * mask_rate)))
    pos = r.choice(len(ids), size=k, replace=False)
    for i in pos:
        new = r.choice(V, p=gen_probs[ids[i]])
        if new != ids[i]:
            ids[i] = new; labels[i] = 1        # 只有真的换掉了才算 replaced
    return ids, labels

seq = rng.integers(0, V, size=20)
gen = make_generator(strength=2.0)
corrupt, rtd_labels = electra_corrupt(seq, gen, seed=1)
print('原始  :', seq[:12])
print('污染后:', corrupt[:12])
print('RTD标签:', rtd_labels[:12], '  (1=被替换)')

# ELECTRA 的输入里根本不存在 [MASK] 这个概念 —— 被换的位置放的是一个真实的词
assert len(rtd_labels) == len(seq), '**每个位置**都有标签 —— 这是 RTD 的核心'
assert rtd_labels.sum() >= 1
assert ((corrupt != seq) == (rtd_labels == 1)).all(), '标签必须与实际替换一致'
print(f'\n信号密度: RTD 每序列 {len(rtd_labels)} 个标签 vs MLM {int(len(seq)*0.15)} 个')
print(f'✅ {len(rtd_labels) / max(1,int(len(seq)*0.15)):.1f}× 的信号密度，且输入分布与下游一致（无 [MASK]）')

### 生成器规模的「难度匹配」：太强太弱都不好

ELECTRA 论文发现生成器规模是判别器的 1/4~1/2 时最优。
用「替换的可分辨性」量化：太弱 → 任务过易；太强 → 任务过难，信号变噪声。

In [ ]:
def discriminability(strength, n=400, seed=0):
    '''用一个最简单的判别器（看词频先验）测量任务难度：
       返回 (被替换比例, 一个朴素判别器的准确率)。'''
    r = np.random.default_rng(seed)
    g = make_generator(strength, seed=seed)
    # 朴素判别器：如果该位置的 token 在语料里是「常见搭配」就判原始
    freq = softmax(r.normal(size=V) * 1.5)
    accs, repl = [], []
    for t in range(n):
        s = r.integers(0, V, size=20)
        c, lab = electra_corrupt(s, g, seed=t)
        pred = (freq[c] < np.median(freq)).astype(int)     # 低频 -> 猜是被替换的
        accs.append((pred == lab).mean()); repl.append(lab.mean())
    return float(np.mean(repl)), float(np.mean(accs))

print(f"{'生成器强度':>11s} {'实际替换率':>10s} {'朴素判别器准确率':>16s} {'说明':<16s}")
rows = []
for st, note in [(-2.0, '太弱: 替换很离谱'), (0.0, '中等'), (2.0, '较强'), (6.0, '太强: 几乎不换')]:
    rr, acc = discriminability(st)
    rows.append((st, rr, acc))
    print(f'{st:>11.1f} {rr:>10.1%} {acc:>16.1%}  {note:<16s}')

rates = [r[1] for r in rows]
assert rates == sorted(rates, reverse=True), '生成器越强（越倾向原词），实际替换率越低'
assert rates[-1] < 0.05, '生成器太强时几乎不产生替换 -> 判别器几乎没有正样本可学'
print('\n✅ 两端都坏：太弱 -> 替换离谱、任务过易、学不到语言知识；')
print('   太强 -> 几乎不替换、正样本稀缺、判别器无信号。中间才是甜点区。')
print('   这与 GAN 里「判别器不能太强」是同一类难度匹配问题 —— 但 ELECTRA **不是** GAN：')
print('   生成器不接收判别器的对抗梯度（文本离散传不回去），两者是协同训练，不是博弈。')

### 损失加权 λ：为什么必须是 50 量级

`L = L_MLM(生成器) + λ · L_RTD(判别器)`。二分类损失的数值尺度远小于 30k 类交叉熵。

In [ ]:
V_REAL = 30522
loss_mlm_scale = math.log(V_REAL)      # 随机初始化时的 MLM 损失 ≈ ln(V)
loss_rtd_scale = math.log(2)           # 随机初始化时的二分类损失 = ln(2)
print(f'初始损失量级: MLM ≈ {loss_mlm_scale:.2f} | RTD ≈ {loss_rtd_scale:.2f}')
print(f'比值 = {loss_mlm_scale / loss_rtd_scale:.1f}')

for lam in [1, 10, 50, 100]:
    share = lam * loss_rtd_scale / (loss_mlm_scale + lam * loss_rtd_scale)
    print(f'  λ={lam:>3d}: RTD 项占总损失 {share:>5.1%}')

share1  = 1  * loss_rtd_scale / (loss_mlm_scale + 1  * loss_rtd_scale)
share50 = 50 * loss_rtd_scale / (loss_mlm_scale + 50 * loss_rtd_scale)
assert share1 < 0.10, 'λ=1 时判别器几乎收不到梯度'
assert share50 > 0.70, 'λ=50 时判别器主导 —— 这正是我们真正要的模型'
print(f'\n✅ λ=1 时 RTD 只占 {share1:.0%}（判别器几乎不更新）；λ=50 时占 {share50:.0%}。')
print('   论文取 50 不是玄学，是让「真正要的那个模型」拿到主要梯度。')

## 3 · DeBERTa：解耦注意力的三项分解

BERT 把位置嵌入**加**到词嵌入上，展开后得到四项。DeBERTa 用相对位置、只保留前三项。
下面把两种实现都写出来，验证分解的正确性。

In [ ]:
def bert_style_attention_scores(X, Pabs, Wq, Wk):
    '''BERT: (x+p)Wq · ((x+p)Wk)^T —— 内容与位置纠缠在一起。'''
    H = X + Pabs
    return (H @ Wq) @ (H @ Wk).T / math.sqrt(Wq.shape[1])

def expand_four_terms(X, Pabs, Wq, Wk):
    '''把上式展开成四项，验证它们之和等于原式。'''
    s = math.sqrt(Wq.shape[1])
    c2c = (X @ Wq) @ (X @ Wk).T / s
    c2p = (X @ Wq) @ (Pabs @ Wk).T / s
    p2c = (Pabs @ Wq) @ (X @ Wk).T / s
    p2p = (Pabs @ Wq) @ (Pabs @ Wk).T / s
    return c2c, c2p, p2c, p2p

n = 8
X = rng.normal(size=(n, D)) * 0.5
Pabs = rng.normal(size=(n, D)) * 0.5
Wq, Wk = rng.normal(size=(D, D)) * 0.2, rng.normal(size=(D, D)) * 0.2

full = bert_style_attention_scores(X, Pabs, Wq, Wk)
c2c, c2p, p2c, p2p = expand_four_terms(X, Pabs, Wq, Wk)
assert np.allclose(full, c2c + c2p + p2c + p2p, atol=1e-10), '四项之和必须等于原式'
print('✅ BERT 注意力 = 内容→内容 + 内容→位置 + 位置→内容 + 位置→位置')
print(f'   各项的分数标准差: c2c {c2c.std():.3f} | c2p {c2p.std():.3f} | '
      f'p2c {p2c.std():.3f} | p2p {p2p.std():.3f}')

In [ ]:
def relative_position_bucket(n, max_rel=4):
    '''相对位置矩阵 rel[i,j] = clip(i-j, -max_rel, max_rel) + max_rel（作为索引）。'''
    idx = np.arange(n)
    rel = np.clip(idx[:, None] - idx[None, :], -max_rel, max_rel) + max_rel
    return rel

def deberta_attention_scores(X, Rel, Wqc, Wkc, Wqr, Wkr, rel_idx):
    '''DeBERTa: 内容→内容 + 内容→位置 + 位置→内容（丢掉 位置→位置）。'''
    s = math.sqrt(Wqc.shape[1]) * math.sqrt(3)      # 论文用 sqrt(3d) 归一化三项
    Qc, Kc = X @ Wqc, X @ Wkc
    Qr, Kr = Rel @ Wqr, Rel @ Wkr
    c2c = Qc @ Kc.T
    c2p = np.take_along_axis(Qc @ Kr.T, rel_idx, axis=1)          # 用相对索引取
    p2c = np.take_along_axis((Kc @ Qr.T), rel_idx.T, axis=1).T
    return (c2c + c2p + p2c) / s

MAX_REL = 4
rel_idx = relative_position_bucket(n, MAX_REL)
Rel = rng.normal(size=(2 * MAX_REL + 1, D)) * 0.5
Wqc, Wkc, Wqr, Wkr = (rng.normal(size=(D, D)) * 0.2 for _ in range(4))
A = deberta_attention_scores(X, Rel, Wqc, Wkc, Wqr, Wkr, rel_idx)
print('DeBERTa 注意力分数矩阵形状:', A.shape)
assert A.shape == (n, n)
assert np.isfinite(A).all()

# 关键性质 ①：相对位置具有平移不变性 —— 距离相同的位置对，位置项贡献相同
print('\n相对位置索引矩阵 rel[i,j] = clip(i-j):')
print(rel_idx)
assert rel_idx[2, 1] == rel_idx[5, 4], '距离相同 -> 相对位置索引相同（平移不变）'
assert rel_idx[0, n-1] == rel_idx[1, n-1], f'超过 max_rel={MAX_REL} 的距离被截断到同一桶'
print(f'\n✅ 相对位置的两个关键性质：平移不变 + 远距离截断。')
print(f'   前者是 DeBERTa 的归纳偏置；后者让位置表只需 {2*MAX_REL+1} 行（BERT 需要 512 行）。')

### 解耦的代价：计算量

In [ ]:
def attn_flops(n, d, mode):
    '''注意力分数的相对计算量（只数矩阵乘）。'''
    base = n * n * d
    return {'bert': base, 'deberta': 3 * base}[mode]

for n_ in [128, 512]:
    b, db = attn_flops(n_, 768, 'bert'), attn_flops(n_, 768, 'deberta')
    print(f'序列长 {n_:>4d}: BERT {b:.2e} | DeBERTa {db:.2e} | {db/b:.1f}×')
assert attn_flops(512, 768, 'deberta') == 3 * attn_flops(512, 768, 'bert')
print('\n✅ 三项分解 = 约 3× 的注意力分数计算（论文实测端到端约 1.5-2× 训练开销）。')
print('   另一个隐性代价：非标准 QK^T，**与 FlashAttention 等 kernel 不兼容**。')

## 4 · ALBERT：参数量 ≠ 效率

跨层共享把参数压掉 89%，但**推理 FLOPs 与延迟完全不变**——层数没少。

In [ ]:
def params_and_flops(vocab, H, n_layers, E=None, share_layers=False, seq_len=512):
    '''E: 嵌入分解的中间维度（None 表示不分解）。返回 (参数量, 每序列前向 FLOPs)。'''
    emb = vocab * H if E is None else vocab * E + E * H
    per_layer = 4 * H * H + 2 * H * (4 * H)              # QKVO + FFN
    layer_params = per_layer if share_layers else per_layer * n_layers
    params = emb + layer_params
    # FLOPs 与是否共享**无关**：还是要跑 n_layers 次
    flops = n_layers * (2 * seq_len * per_layer + 2 * seq_len * seq_len * H)
    return params, flops

configs = [
    ('BERT-base',        dict(vocab=30000, H=768,  n_layers=12)),
    ('ALBERT-base',      dict(vocab=30000, H=768,  n_layers=12, E=128, share_layers=True)),
    ('BERT-large',       dict(vocab=30000, H=1024, n_layers=24)),
    ('ALBERT-xxlarge',   dict(vocab=30000, H=4096, n_layers=12, E=128, share_layers=True)),
]
print(f"{'模型':<18s} {'参数(M)':>9s} {'前向GFLOPs':>12s} {'参数/BERT-base':>15s} {'FLOPs/BERT-base':>16s}")
p0, f0 = params_and_flops(vocab=30000, H=768, n_layers=12)
for name, cfg in configs:
    p, f = params_and_flops(**cfg)
    print(f'{name:<18s} {p/1e6:>9.1f} {f/1e9:>12.1f} {p/p0:>15.2f}× {f/f0:>15.2f}×')

p_bert, f_bert = params_and_flops(vocab=30000, H=768, n_layers=12)
p_alb,  f_alb  = params_and_flops(vocab=30000, H=768, n_layers=12, E=128, share_layers=True)
assert p_alb < 0.2 * p_bert, 'ALBERT-base 参数应少 80% 以上'
assert abs(f_alb - f_bert) < 1e-9, '**FLOPs 完全相同** —— 层数没少，还是要算 12 次'
p_xx, f_xx = params_and_flops(vocab=30000, H=4096, n_layers=12, E=128, share_layers=True)
assert f_xx > 3 * f_bert, 'ALBERT-xxlarge 虽然参数不多，但每层更宽 -> 比 BERT-base 慢数倍'
print(f'\n✅ ALBERT-base: 参数 {p_alb/p_bert:.0%}，FLOPs {f_alb/f_bert:.0%} —— **一点都没省**。')
print(f'   ALBERT-xxlarge: 参数才 {p_xx/1e6:.0f}M，但 FLOPs 是 BERT-base 的 {f_xx/f_bert:.1f}×。')
print('   结论：**参数量是很差的效率代理指标**，要看 FLOPs 与实测延迟。')

### 嵌入分解省在哪

In [ ]:
V_REAL, H = 30000, 768
plain = V_REAL * H
for E in [64, 128, 256]:
    fact = V_REAL * E + E * H
    print(f'E={E:>4d}: 嵌入参数 {fact/1e6:>5.1f}M (原 {plain/1e6:.1f}M, 省 {(1-fact/plain):.0%})')
assert V_REAL * 128 + 128 * H < 0.25 * plain, 'E=128 应省掉 75% 以上的嵌入参数'
print(f'\n✅ 嵌入表在 BERT-base 里占 {plain/(plain + 12*(4*H*H+8*H*H)):.0%}，分解后大幅缩小。')
print('   这一项是**真省**（参数与显存都省），只是它省的不是瓶颈。')

## ✏️ 练习 1：动态掩码的有效样本量

实现 `effective_samples(n_seqs, n_epochs, static_copies=None, L=20, k=3, seed=0)`：
统计模型在整个训练过程中见到的**不同 (序列, 掩码模式) 组合**数量。
`static_copies=None` 表示动态掩码（每 epoch 现采）；给整数表示静态副本数。

In [ ]:
def effective_samples(n_seqs, n_epochs, static_copies=None, L=20, k=3, seed=0):
    # TODO: 静态：每条序列预生成 static_copies 个模式，训练时循环取；
    #       动态：每条序列每个 epoch 现采一个模式。
    #       返回 len({(seq_id, frozenset(mask_positions))})
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
s = effective_samples(100, 40, static_copies=10)
d = effective_samples(100, 40, static_copies=None)
assert s <= 100 * 10, '静态：最多 n_seqs × copies 种组合'
assert d > s, '动态应见到更多组合'
assert d <= 100 * 40, '动态：最多 n_seqs × epochs 种组合'
# 副本越多，静态越接近动态
s20 = effective_samples(100, 40, static_copies=20)
assert s20 > s, '更多静态副本 -> 更多组合'
print(f'静态10份: {s:,} 种 | 静态20份: {s20:,} 种 | 动态40轮: {d:,} 种')
print('✅ 练习 1 通过：动态掩码是零成本的数据增强')

## ✏️ 练习 2：信号密度 × 信息量

实现 `objective_efficiency(objective, L, vocab, mask_rate=0.15)`：
返回 `(每序列预测数, 每次预测的最大信息量bit, 每序列信息上限bit)`。
- `'CLM'`：L 次预测，每次 `log2(vocab)` bit
- `'MLM'`：`L*mask_rate` 次，每次 `log2(vocab)` bit
- `'RTD'`：L 次，每次 1 bit
- `'RTD-multi'`：L 次，每次 `log2(k+1)` bit（k 类替换 + 原始），取 `k=3`

In [ ]:
def objective_efficiency(objective, L, vocab, mask_rate=0.15, k=3):
    # TODO: 返回 (n_pred, bits_per_pred, total_bits)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
L_, VV = 512, 30522
clm = objective_efficiency('CLM', L_, VV)
mlm = objective_efficiency('MLM', L_, VV)
rtd = objective_efficiency('RTD', L_, VV)
rtm = objective_efficiency('RTD-multi', L_, VV)
for name, r in [('CLM', clm), ('MLM', mlm), ('RTD', rtd), ('RTD-multi', rtm)]:
    print(f'{name:<10s} 预测数 {r[0]:>6.0f} | 每次 {r[1]:>5.1f} bit | 合计 {r[2]:>9.0f} bit')
assert clm[0] == L_ and abs(mlm[0] - L_ * 0.15) < 1
assert rtd[1] == 1.0 and rtd[0] == L_
assert mlm[2] > rtd[2], 'MLM 单次信息量大，总信息上限反而更高'
assert rtm[2] > rtd[2], '多分类替换检测的信息量介于两者之间'
assert clm[2] > mlm[2], 'CLM 在两个维度上都不吃亏（但表示是单向的）'
print('\n✅ 练习 2 通过：ELECTRA 的实际优势 ~4×，而非朴素密度比的 6.7× —— 账要算细')

## ✏️ 练习 3：相对位置分桶

实现 `rel_bucket(i, j, max_rel)`：返回相对位置 `i-j` 的桶索引，范围 `[0, 2*max_rel]`。
再实现 `bucket_matrix(n, max_rel)` 返回整个 `(n,n)` 索引矩阵。
必须满足：平移不变（`rel_bucket(5,4)==rel_bucket(2,1)`）、超距截断、对角线为 `max_rel`。

In [ ]:
def rel_bucket(i, j, max_rel):
    # TODO
    raise NotImplementedError

def bucket_matrix(n, max_rel):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert rel_bucket(5, 4, 4) == rel_bucket(2, 1, 4), '平移不变'
assert rel_bucket(3, 3, 4) == 4, '对角线（距离 0）应落在中间桶 max_rel'
assert rel_bucket(0, 100, 4) == 0, '极远的左向距离被截断到最小桶'
assert rel_bucket(100, 0, 4) == 8, '极远的右向距离被截断到最大桶'
M = bucket_matrix(6, 4)
assert M.shape == (6, 6)
assert (np.diag(M) == 4).all()
assert M.min() >= 0 and M.max() <= 8, '索引必须落在 [0, 2*max_rel]'
assert np.array_equal(M, bucket_matrix(6, 4)), '应是确定性的'
print(bucket_matrix(6, 3))
print('✅ 练习 3 通过：位置表只需 2*max_rel+1 行，而 BERT 需要 max_position_embeddings 行')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def effective_samples(n_seqs, n_epochs, static_copies=None, L=20, k=3, seed=0):
    r = np.random.default_rng(seed)
    seen = set()
    if static_copies is not None:
        pats = {s: [frozenset(r.choice(L, size=k, replace=False)) for _ in range(static_copies)]
                for s in range(n_seqs)}
        for ep in range(n_epochs):
            for s in range(n_seqs):
                seen.add((s, pats[s][ep % static_copies]))
    else:
        for ep in range(n_epochs):
            for s in range(n_seqs):
                seen.add((s, frozenset(r.choice(L, size=k, replace=False))))
    return len(seen)

In [ ]:
# 练习 2 参考答案
def objective_efficiency(objective, L, vocab, mask_rate=0.15, k=3):
    if objective == 'CLM':        n, bits = L, math.log2(vocab)
    elif objective == 'MLM':      n, bits = L * mask_rate, math.log2(vocab)
    elif objective == 'RTD':      n, bits = L, 1.0
    elif objective == 'RTD-multi': n, bits = L, math.log2(k + 1)
    else: raise ValueError(objective)
    return n, bits, n * bits

In [ ]:
# 练习 3 参考答案
def rel_bucket(i, j, max_rel):
    return int(np.clip(i - j, -max_rel, max_rel)) + max_rel

def bucket_matrix(n, max_rel):
    idx = np.arange(n)
    return np.clip(idx[:, None] - idx[None, :], -max_rel, max_rel) + max_rel

---
## 🧪 真实数据胶囊：四种改进的效率前沿

把四个工作放到「算力 vs 分数」的平面上。你会看到它们**不在同一个坐标轴上竞争**。
（数字取自各论文报告的量级。）

In [ ]:
models = [
    # (名称, 参数M, 预训练算力(相对BERT-base), GLUE量级, 推理FLOPs(相对))
    ('BERT-base',      110, 1.00, 79.0, 1.00),
    ('RoBERTa-base',   125, 4.00, 86.0, 1.00),
    ('ELECTRA-base',   110, 0.25, 86.0, 1.00),
    ('ALBERT-base',     12, 1.00, 80.0, 1.00),
    ('DeBERTa-base',   140, 1.50, 88.0, 1.60),
]
print(f"{'模型':<16s} {'参数M':>7s} {'预训练算力':>10s} {'GLUE':>6s} {'推理FLOPs':>10s} {'分/算力':>8s}")
for name, p, c, g, f in models:
    print(f'{name:<16s} {p:>7d} {c:>10.2f}× {g:>6.1f} {f:>9.2f}× {g/c:>8.1f}')

by_name = {m[0]: m for m in models}
# ELECTRA：同分数、1/16 算力
assert by_name['ELECTRA-base'][3] == by_name['RoBERTa-base'][3]
assert by_name['ELECTRA-base'][2] < by_name['RoBERTa-base'][2] / 10
# ALBERT：参数少但推理 FLOPs 一样
assert by_name['ALBERT-base'][1] < 0.2 * by_name['BERT-base'][1]
assert by_name['ALBERT-base'][4] == by_name['BERT-base'][4]
# DeBERTa：分数最高但推理更贵
assert by_name['DeBERTa-base'][3] == max(m[3] for m in models)
assert by_name['DeBERTa-base'][4] > 1.0
print('\n✅ 三种「更好」是三个不同方向：')
print('   RoBERTa/DeBERTa = 用更多算力换更高分数（前沿右上移）')
print('   ELECTRA         = 用更少算力达到同样分数（前沿左移）← 真正的效率改进')
print('   ALBERT          = 用同样算力换更小的权重文件（第三个维度）')
print('   问「谁最好」是坏问题；问「我被什么卡住」才对。')

**🧪 胶囊练习**：实现 `recommend(constraint)`：给定约束返回推荐方案与理由。
`constraint ∈ {'compute', 'accuracy', 'latency', 'model_size'}`，
分别返回 `'ELECTRA-RTD'` / `'DeBERTa-v3'` / `'distill-to-smaller'` / `'ALBERT-factorized-embedding'`。

In [ ]:
def recommend(constraint):
    # TODO: 按上述映射返回字符串
    raise NotImplementedError

In [ ]:
# 自测
assert recommend('compute')    == 'ELECTRA-RTD'
assert recommend('accuracy')   == 'DeBERTa-v3'
assert recommend('latency')    == 'distill-to-smaller'
assert recommend('model_size') == 'ALBERT-factorized-embedding'
for c in ['compute', 'accuracy', 'latency', 'model_size']:
    print(f'{c:<12s} -> {recommend(c)}')
print('\n⚠️  注意 latency 的答案**不是** ALBERT —— 参数共享不省延迟（本模块已验证）。')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def recommend(constraint):
    return {'compute': 'ELECTRA-RTD',
            'accuracy': 'DeBERTa-v3',
            'latency': 'distill-to-smaller',
            'model_size': 'ALBERT-factorized-embedding'}[constraint]

---
## 🔧 旁注：真实库里这些对应什么

- **动态掩码** → `DataCollatorForLanguageModeling` 在每个 batch 现场掩码（HF 默认就是动态的）。
- **ELECTRA** → `transformers.ElectraForPreTraining`（判别器）+ `ElectraForMaskedLM`（生成器）；下游只用判别器。
- **DeBERTa 解耦注意力** → `DebertaV2Model`；`config.pos_att_type=["p2c","c2p"]` 就是那两项。
- **ALBERT 参数共享** → `AlbertModel` 内部只有一份 `AlbertLayer`，循环调用 `num_hidden_layers` 次。
- **嵌入分解** → `config.embedding_size`（128）与 `config.hidden_size`（768）分离。
- **选型** → 今天做编码器任务，起点建议是 `microsoft/deberta-v3-base` 或现代化的 BERT 变体，**不是** `bert-base-uncased`。

怎么真正加载和微调这些模型，见 **C50**。

### 小结
- **RoBERTa 零架构改动、只调配方，就提升 7 分**——BERT 是严重欠训练的。教训：宣布架构创新前先确认 baseline 训练充分。
- **动态掩码是免费的数据增强**；训练时的随机性能显著扩充有效样本量。
- **ELECTRA 的 RTD 把信号密度从 0.15 拉回 1.0**，且输入无 `[MASK]`。它不是 GAN（无对抗梯度）；生成器要「小」是难度匹配；λ=50 是为了让判别器拿到主要梯度。
- **DeBERTa 解耦内容与位置**，用相对位置获得平移不变与小位置表，代价是约 3× 注意力分数计算且与 FlashAttention 不兼容。
- **ALBERT 证明了参数量 ≠ 效率**：跨层共享省 89% 参数但 FLOPs 一点不省；它的持久贡献是 SOP 与嵌入分解。
- 四个维度**几乎正交、可以叠加**（DeBERTa-v3 = 解耦注意力 + RTD + 充分训练）。

下一站：**模块 03 · 下游微调三范式** —— 预训练好的表示，怎么接到分类、标注、抽取三类任务上。